In [11]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append(
    "./modules/python-utils:./modules/ai-utils"
)

In [12]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import numpy as np

In [13]:
from sj_ai_utils.asr.whisper_utils import segments_to_sclite_trn

In [14]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/1688/142285/"
DESTINATION = "/workspaces/dev/datasets/LibriSpeechASRcorpus/sclient/test-other/1688/142285"
HYP_PATH =  "/workspaces/dev/datasets/LibriSpeechASRcorpus/sclient/test-other/1688/142285/1688_142285.hyp.trn"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [15]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="int8")

In [16]:
src = Path(SOURCE)
dest = Path(DESTINATION)
hyp_path = Path(HYP_PATH)
dest.mkdir(parents=True, exist_ok=True)

In [17]:
audio_files = [f for f in src.glob("*.flac")]
audios = [{"path": audio, "audio": librosa.load(audio, sr=SAMPLE_RATE)[0]} for audio in audio_files]

In [18]:
def transcribe(audio:np.ndarray):
    segments, info = model.transcribe(audio)
    return segments

In [19]:
trn_content = []
for audio in audios:
    segments = transcribe(audio["audio"])
    trn_content.append(segments_to_sclite_trn(audio["path"].name, segments))

trn = "\n".join(trn_content)
with open(hyp_path, "w") as hyp_file:
    hyp_file.write(trn)